## Bronze — População por Município (IBGE via Base dos Dados)

**Fonte:** Base dos Dados — tabela `br_ibge_populacao.municipio` ([População Brasileira](https://basedosdados.org/dataset/d30222ad-7a5c-4778-a1ec-f0785371d1ca?table=0c279444-165b-41da-92cd-50fd7e66baa1))

- Licença: Base dos Dados (CC-BY-4.0) — republicação dos dados públicos do IBGE (estimativas de população).
- Área responsável pela fonte original: IBGE (estimativas com referência em 1º de julho; anos de censo/contagem usam o total recenseado).
- **Arquivo origem:** `/Volumes/workspace/raw/basedosdados/br_ibge_populacao_municipio.csv` (UTF-8), cobertura **1991 a 2025**.
- **Formato do arquivo:** header na 1ª linha (`ano, sigla_uf, id_municipio, populacao`), sem linhas de controle.
- **Grão:** 1 linha por município x ano.
- **Linhagem:** download Base dos Dados → CSV no Volume → `workspace.bronze.estimativa_populacional`.

In [0]:
%run ../shared/_setup

In [0]:
from pyspark.sql import functions as F
from data_pipeline import read_csv, normalizar_colunas, save_table, add_column_comments
from catalogo.populacao import BRONZE_ESTIMATIVA_POPULACAO_COMMENTS

In [0]:
FILE_PATH = "/Volumes/workspace/raw/basedosdados/br_ibge_populacao_municipio.csv"
TABLE_NAME = "workspace.bronze.estimativa_populacional"
CSV_ENCODING = "UTF-8"
CSV_DELIMITER = ","

In [0]:
# Arquivo com header na 1ª linha (ano,sigla_uf,id_municipio,populacao) — leitura direta
df = read_csv(
    spark,
    FILE_PATH,
    delimiter=CSV_DELIMITER,
    encoding=CSV_ENCODING,
)
df = normalizar_colunas(df)
print(f"Total de linhas: {df.count():,} | colunas: {df.columns}")
display(df.limit(10))

In [0]:
save_table(df, TABLE_NAME)
add_column_comments(
    spark,
    TABLE_NAME,
    BRONZE_ESTIMATIVA_POPULACAO_COMMENTS
)

In [0]:
total = spark.table(TABLE_NAME).count()
distintos = spark.table(TABLE_NAME).distinct().count()
print(f"Total: {total:,} | Linhas únicas: {distintos:,} | Duplicatas: {total - distintos:,}")
display(spark.sql(f"SELECT ano, count(*) AS qtd_municipios FROM {TABLE_NAME} GROUP BY ano ORDER BY ano"))
display(spark.sql(f"SELECT sigla_uf, count(*) AS qtd FROM {TABLE_NAME} GROUP BY sigla_uf ORDER BY sigla_uf"))